# Capital One — Data Scientist (Associate / New Grad) Practice Assessment

**Budget: 80 minutes.** Set a timer and do not stop it. Finishing everything is not the goal;
producing correct, defensible work under time pressure is.

| Section | Tasks | Budget | Why it's here |
|---|---|---|---|
| 1 — pandas | P1–P4 | 20 min | Data collection & processing: the messy-extract-to-clean-table loop |
| 2 — SQL | S1–S5 | 30 min | Window functions, CTEs, self-joins on a card portfolio |
| 3 — Applied ML | M1–M4 | 30 min | Leakage, honest validation, and a business-cost decision |

### Rules that make this useful

1. **One pass, no going back.** When a section's budget is up, move on even if it's unfinished.
2. **Docs are allowed. Answers are not.** pandas/sklearn/SQLite docs are fine — that mirrors the real thing.
   Do not look at `SOLUTIONS.md` until the timer stops.
3. **Run `check(...)` as you go.** It tells you *that* something is wrong and roughly where, without
   giving away the answer. Treat a FAIL like a failing hidden test in the real assessment: reread the
   spec, find your own bug.
4. **Answer shapes are specified exactly.** Column names, sort order and rounding are part of the task,
   the same way they are when a hidden test is grading you.
5. **M4 is written, not coded.** Do not skip it. At Capital One the open-ended reasoning carries as much
   weight as the code, and it is the part people prepare least.

### Scoring yourself

Run `score()` at the end. Rough calibration for this set:

- **≥ 85%** with time to spare — you're in good shape.
- **65–85%** — competitive; find your slow section and drill it.
- **< 65%** — you have a specific gap, not a general one. The per-section breakdown will name it.

---

## Setup

Run this once. It locates the problem set on its own, so this notebook works both in place and as a
copy under `attempts/`.

In [91]:
import os, sys, sqlite3, time
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

SET_NAME = "capital-one-ds"

def _find_set_root(name):
    """Works whether this notebook sits in the set folder or in attempts/."""
    start = os.path.abspath("")
    # already inside the set folder (or a child of it)?
    p = start
    while True:
        if all(os.path.exists(os.path.join(p, f)) for f in ("grader.py", "make_key.py")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    # otherwise walk up to the repo root and look under sets/
    p = start
    while True:
        cand = os.path.join(p, "sets", name)
        if os.path.isdir(cand):
            return cand
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    raise RuntimeError(
        f"Could not find the '{name}' problem set. Open this notebook from inside the repo "
        f"(either sets/{name}/ or attempts/<date>-{name}/)."
    )

SET_ROOT = _find_set_root(SET_NAME)
DATA = os.path.join(SET_ROOT, "data")
if SET_ROOT not in sys.path:
    sys.path.insert(0, SET_ROOT)

if not os.path.isdir(DATA):
    raise RuntimeError(f"No data yet. Run:  {os.path.join(SET_ROOT, 'bootstrap.sh')}")

CON = sqlite3.connect(os.path.join(DATA, "warehouse.db"))

def q(sql: str) -> pd.DataFrame:
    """Run SQL against the warehouse and return a DataFrame."""
    if not sql.strip():
        print("(empty query -- write your SQL between the triple quotes)")
        return pd.DataFrame()
    return pd.read_sql_query(sql, CON)

from grader import check, score

_T0 = time.time()
def elapsed(label=""):
    print(f"[{(time.time() - _T0) / 60:5.1f} min] {label}")

with open(os.path.join(DATA, "SEED")) as _f:
    SEED = _f.read().strip()

print(f"set:   {SET_ROOT}")
print(f"seed:  {SEED}   <- record this in your NOTES.md")
print("tables:", q("SELECT name FROM sqlite_master WHERE type='table'").name.tolist())
elapsed("timer started")

set:   c:\Users\rocco\OneDrive\Documents\GitHub\ds_practice\sets\capital-one-ds
seed:  20260917   <- record this in your NOTES.md
tables: ['customers', 'accounts', 'transactions', 'statements']
[  0.0 min] timer started


---
# Section 1 — pandas  ·  budget 20 minutes

You are handed a raw extract of 2025 card transactions plus two reference files.

| file | grain | notes |
|---|---|---|
| `data/transactions_raw.csv` | one row per transaction | **raw system extract — it is dirty** |
| `data/accounts.csv` | one row per card account | `account_id`, `customer_id`, `product`, `open_date`, `credit_limit`, `apr`, `status` |
| `data/customers.csv` | one row per customer | `customer_id`, `signup_date`, `state`, `age`, `segment`, `annual_income`, `credit_score_at_signup` |

A customer can hold more than one account. Every transaction in the extract falls in calendar 2025.

Start by looking at the raw file before you write anything.

In [64]:
raw = pd.read_csv(f"{DATA}/transactions_raw.csv")
print(raw.shape)
print(raw.dtypes)
raw.head(8)

(113111, 8)
transaction_id         str
account_id             str
txn_ts                 str
amount                 str
merchant_category      str
merchant_id            str
channel                str
is_disputed          int64
dtype: object


,transaction_id,account_id,txn_ts,amount,merchant_category,merchant_id,channel,is_disputed
0,T500078034,A2000874,2025-09-25 04:59:28,17.26,Rideshare,M8430,online,0
1,T500092039,A2004520,2025-08-17 11:23:17,111.74,Utilities,M4426,online,0
2,T500110395,A2000055,2025-02-15 02:27:57,83.31,NaN,M7553,online,0
3,T500096581,A2001216,2025-06-09 12:46:48,19.23,Groceries,M8807,in_store,0
4,T500112074,A2000539,2025-05-10 23:50:50,$26.22,GROCERIES,M6617,in_store,0
5,T500071384,A2001084,2025-11-29 20:04:22,446.36,Travel,M6956,in_store,0
6,T500008289,A2003907,2025-09-16 19:28:29,92.14,Gas,M8646,online,0
7,T500108236,A2000817,2025-03-23 14:25:28,120.24,RETAIL,M4127,online,0


### P1 — Clean the extract  *(~6 min)*

Three things are wrong with `transactions_raw.csv`:

- **`amount` is text**, with inconsistent formatting: `1234.56`, `$1,234.56`, `1,234.56`,
  values padded with whitespace, and **accounting-style negatives for refunds**, e.g. `($45.20)`
  means −45.20.
- **`merchant_category` has inconsistent casing and stray whitespace** (`Groceries`, `groceries `,
  `GROCERIES` are the same category). Some values are genuinely missing — leave those missing.
- **The extract contains fully duplicated rows** (every column identical, including `transaction_id`).

Produce a cleaned frame where `amount` is a float, `merchant_category` is lowercase and stripped,
`txn_ts` is a datetime, and exact duplicate rows are gone.

**Assign `p1` as a dict with exactly these keys:**

| key | type | meaning |
|---|---|---|
| `n_rows` | int | rows remaining after dropping exact duplicates |
| `n_categories` | int | distinct **non-null** normalized categories |
| `net_amount` | float | sum of all cleaned amounts (purchases **and** refunds), rounded to 2dp |
| `refund_count` | int | rows with a negative amount |

Keep your cleaned frame in a variable — P2, P3 and P4 all build on it.

In [65]:
def clean_amt(series):
    temp = series.astype("string").str.strip()
    negatives = temp.str.startswith("(") & temp.str.endswith(")")
    temp = temp.str.replace(r"[()$,\s]", "", regex=True)
    output = pd.to_numeric(temp, errors ="coerce")
    return output.where(~negatives, -output).astype("float64")

In [102]:
clean = raw.drop_duplicates().copy()
clean["amount"] = clean_amt(clean["amount"])
clean["merchant_category"] = clean["merchant_category"].astype("string").str.lower().str.strip()
clean["txn_ts"] = pd.to_datetime(clean["txn_ts"])

In [84]:
# --- P1 ---
p1 = {}
p1 = {
    "n_rows": int(len(clean)),
    "n_categories": int(clean["merchant_category"].dropna().nunique()),
    "net_amount": float(round(clean["amount"].sum(), 2)),
    "refund_count": int((clean["amount"]<0).sum()),
}
check("P1", p1)

--- P1 ----------------------------------------------------------
PASS P1.n_rows
PASS P1.n_categories
PASS P1.net_amount
PASS P1.refund_count
    P1 score: 100%



### P2 — Spend by segment and product  *(~5 min)*

Using the **cleaned** transactions joined to `accounts` and `customers`, and counting
**purchases only** (`amount > 0`; refunds are excluded entirely):

For each `(segment, product)` pair compute
- `n_accounts` — distinct accounts with at least one purchase,
- `total_spend` — sum of purchase amounts, rounded to 2dp,
- `spend_per_account` — `total_spend / n_accounts`, rounded to 2dp.

**Assign `p2` as a DataFrame** with columns `["segment", "product", "n_accounts", "total_spend",
"spend_per_account"]`, sorted by `segment` then `product` ascending, with a clean `RangeIndex`.

In [93]:
# --- P2 ---
p2 = None
acc = pd.read_csv(f"{DATA}/accounts.csv")
cus = pd.read_csv(f"{DATA}/customers.csv")

purchases = clean[clean["amount"]>0]
merged = (purchases.merge(acc[["account_id", "customer_id", "product"]], on="account_id", how="inner")
                    .merge(cus[["customer_id", "segment"]], on="customer_id", how="inner"))

merged.head()

,transaction_id,account_id,txn_ts,amount,merchant_category,merchant_id,channel,is_disputed,customer_id,product,segment
0,T500078034,A2000874,2025-09-25 04:59:28,17.2600,rideshare,M8430,online,0,C100707,PlatinumSecured,Mass
1,T500092039,A2004520,2025-08-17 11:23:17,111.7400,utilities,M4426,online,0,C103696,Quicksilver,Student
2,T500110395,A2000055,2025-02-15 02:27:57,83.3100,<NA>,M7553,online,0,C100043,Quicksilver,Mass
3,T500096581,A2001216,2025-06-09 12:46:48,19.2300,groceries,M8807,in_store,0,C100989,Savor,Mass
4,T500112074,A2000539,2025-05-10 23:50:50,26.2200,groceries,M6617,in_store,0,C100440,PlatinumSecured,Student


In [94]:
p2 = (merged.groupby(["segment", "product"], as_index=False)
            .agg(n_accounts=("account_id", "nunique"),
                 total_spend=("amount", "sum")))

p2["total_spend"] = p2["total_spend"].round(2)

p2["spend_per_account"] = (p2["total_spend"]/p2["n_accounts"]).round(2)

p2 = p2.sort_values(["segment", "product"]).reset_index(drop=True)
p2

,segment,product,n_accounts,total_spend,spend_per_account
0,Affluent,PlatinumSecured,14,"55,566.4300","3,969.0300"
1,Affluent,Quicksilver,265,"984,011.1000","3,713.2500"
2,Affluent,Savor,170,"635,439.8900","3,737.8800"
3,Affluent,Venture,497,"1,859,770.0000","3,741.9900"
4,Mass,PlatinumSecured,315,"413,047.1600","1,311.2600"
5,Mass,Quicksilver,1147,"1,463,703.5400","1,276.1100"
6,Mass,Savor,724,"927,624.7600","1,281.2500"
7,Mass,Venture,491,"625,948.7500","1,274.8400"
8,SmallBusiness,PlatinumSecured,23,"45,862.4600","1,994.0200"
9,SmallBusiness,Quicksilver,203,"425,793.7000","2,097.5100"


In [90]:
check("P2", p2)

--- P2 ----------------------------------------------------------
PASS P2: 16 rows, all values match
    P2 score: 100%



### P3 — Trailing-30-day velocity flag  *(~6 min)*

Fraud and credit-line teams watch **spend velocity against the line**, not single transactions.

Definition, precisely:
1. Keep purchases only (`amount > 0`).
2. Roll each account's purchases up to a **daily total**.
3. For each account, on every day that account had a purchase, compute the **trailing 30-day
   purchase total**: the window is the 30 calendar days ending on and including that day.
4. An account is **flagged** on the first day where that trailing total **exceeds its `credit_limit`**.

**Assign `p3` as a dict:**

| key | type | meaning |
|---|---|---|
| `n_accounts_flagged` | int | accounts flagged at least once in 2025 |
| `n_flagged_in_q1` | int | of those, how many were **first** flagged before 2025-04-01 |
| `max_trailing30_ratio` | float | the largest `trailing_30_day_total / credit_limit` seen across all accounts and days, rounded to 2dp |

*Hint if you're stuck on mechanics: `df.set_index(date_col).groupby(key)[value].rolling("30D").sum()`.*

In [95]:
purchases = clean[clean["amount"]>0].copy()
purchases["date"] = purchases["txn_ts"].dt.normalize()

daily_total = (purchases.groupby(["account_id", "date"], as_index=False)["amount"].sum()
                .sort_values(["account_id", "date"]))

daily_total

,account_id,date,amount
0,A2000001,2025-03-28,83.9400
1,A2000001,2025-04-08,166.9300
2,A2000001,2025-04-16,218.5700
3,A2000001,2025-05-09,17.5200
4,A2000001,2025-06-13,27.5800
...,...,...,...
104584,A2004892,2025-10-22,44.0400
104585,A2004892,2025-11-07,44.0800
104586,A2004892,2025-11-15,44.5200
104587,A2004892,2025-11-17,106.2800


In [97]:
rolling = (daily_total.set_index("date")
            .groupby("account_id")["amount"]
            .rolling("30D").sum()
            .reset_index(name="trailing30"))

rolling = rolling.merge(acc[["account_id", "credit_limit"]], on="account_id", how="left")
rolling["ratio"] = rolling["trailing30"]/rolling["credit_limit"]

first_flag = rolling[rolling["ratio"] > 1.0].groupby("account_id", as_index=False)["date"].min()

rolling

,account_id,date,trailing30,credit_limit,ratio
0,A2000001,2025-03-28,83.9400,"1,200.0000",0.0699
1,A2000001,2025-04-08,250.8700,"1,200.0000",0.2091
2,A2000001,2025-04-16,469.4400,"1,200.0000",0.3912
3,A2000001,2025-05-09,236.0900,"1,200.0000",0.1967
4,A2000001,2025-06-13,27.5800,"1,200.0000",0.0230
...,...,...,...,...,...
104584,A2004892,2025-10-22,63.8500,600.0000,0.1064
104585,A2004892,2025-11-07,88.1200,600.0000,0.1469
104586,A2004892,2025-11-15,132.6400,600.0000,0.2211
104587,A2004892,2025-11-17,238.9200,600.0000,0.3982


In [98]:
# --- P3 ---

p3 = {
    "n_accounts_flagged": int(len(first_flag)),
    "n_flagged_in_q1": int((first_flag["date"]<"2025-04-01").sum()),
    "max_trailing30_ratio": float(round(rolling["ratio"].max(), 2)),
}
check("P3", p3)

--- P3 ----------------------------------------------------------
PASS P3.n_accounts_flagged
PASS P3.n_flagged_in_q1
PASS P3.max_trailing30_ratio
    P3 score: 100%



### P4 — Share-of-wallet concentration  *(~3 min)*

Aggregate to the **customer** level (a customer's accounts combine). Use purchases only
(`amount > 0`) **with a non-null category**.

- A customer is **eligible** if they have **at least 20** such purchases.
- For each eligible customer, compute the share of their total purchase spend that falls in their
  single largest category (break ties alphabetically by category name).
- A customer is **concentrated** if that top share is **≥ 0.35**.

**Assign `p4` as a dict:**

| key | type | meaning |
|---|---|---|
| `n_eligible` | int | eligible customers |
| `n_concentrated` | int | eligible customers with top share ≥ 0.35 |
| `top_category` | str | the category that is the #1 category for the most eligible customers (lowercase) |
| `mean_top_share` | float | mean top-category share across eligible customers, rounded to 4dp |

In [ ]:
purchases_nonna = clean[(clean["amount"]>0) & clean["merchant_category"].notna()]
merged = purchases_nonna.merge(acc[["account_id","customer_id"]], on="account_id", how="inner")
merged.head()

,transaction_id,account_id,txn_ts,amount,merchant_category,merchant_id,channel,is_disputed,customer_id
0,T500078034,A2000874,2025-09-25 04:59:28,17.2600,rideshare,M8430,online,0,C100707
1,T500092039,A2004520,2025-08-17 11:23:17,111.7400,utilities,M4426,online,0,C103696
2,T500096581,A2001216,2025-06-09 12:46:48,19.2300,groceries,M8807,in_store,0,C100989
3,T500112074,A2000539,2025-05-10 23:50:50,26.2200,groceries,M6617,in_store,0,C100440
4,T500071384,A2001084,2025-11-29 20:04:22,446.3600,travel,M6956,in_store,0,C100877


In [116]:
num_purchases = merged.groupby("customer_id").size()
eligible = num_purchases[num_purchases>=20].index
# num_purchases.head()
# eligible
merged_new = merged[merged["customer_id"].isin(eligible)]
merged_new

,transaction_id,account_id,txn_ts,amount,merchant_category,merchant_id,channel,is_disputed,customer_id
2,T500096581,A2001216,2025-06-09 12:46:48,19.2300,groceries,M8807,in_store,0,C100989
4,T500071384,A2001084,2025-11-29 20:04:22,446.3600,travel,M6956,in_store,0,C100877
5,T500008289,A2003907,2025-09-16 19:28:29,92.1400,gas,M8646,online,0,C103199
6,T500108236,A2000817,2025-03-23 14:25:28,120.2400,retail,M4127,online,0,C100659
8,T500112008,A2001360,2025-03-06 14:53:56,5.6200,streaming,M1062,in_store,0,C101104
...,...,...,...,...,...,...,...,...,...
106897,T500066455,A2003807,2025-02-04 03:33:33,85.2700,retail,M7120,in_store,0,C103119
106898,T500053459,A2003448,2025-03-20 06:35:10,39.9000,groceries,M8064,online,0,C102813
106900,T500010742,A2003009,2025-10-28 01:26:43,115.6000,healthcare,M2297,in_store,0,C102454
106901,T500049689,A2001512,2025-06-29 13:01:21,24.4600,pharmacy,M2832,online,0,C101230


In [121]:
cat = merged_new.groupby(["customer_id", "merchant_category"], as_index=False)["amount"].sum()
# cat.head()
total = cat.groupby("customer_id", as_index=False)["amount"].sum().rename(columns={"amount": "total"})
# total.head()
top = (cat.sort_values(["customer_id", "amount", "merchant_category"], 
                        ascending=[True, False, True])
            .groupby("customer_id", as_index=False).first()
            .merge(total, on="customer_id"))

top["share"] = top["amount"]/top["total"]

top.head()

,customer_id,merchant_category,amount,total,share
0,C100001,restaurants,374.9000,"1,234.5200",0.3037
1,C100003,groceries,884.7900,"4,359.9900",0.2029
2,C100004,groceries,855.3200,"2,876.7000",0.2973
3,C100006,travel,"1,799.8600","7,690.5700",0.2340
4,C100007,groceries,"1,195.8400","3,605.8400",0.3316


In [122]:
# --- P4 ---


p4 = {
    "n_eligible": int(len(eligible)),
    "n_concentrated": int((top["share"]>=0.35).sum()),
    "top_category": str(top["merchant_category"].value_counts().idxmax()),
    "mean_top_share": float(round(top["share"].mean(), 4)),
}
check("P4", p4)

--- P4 ----------------------------------------------------------
PASS P4.n_eligible
PASS P4.n_concentrated
PASS P4.top_category
PASS P4.mean_top_share
    P4 score: 100%



In [40]:
elapsed("end of Section 1 -- target was 20 min")

[ 33.1 min] end of Section 1 -- target was 20 min


---
# Section 2 — SQL  ·  budget 30 minutes

`data/warehouse.db` is SQLite. Unlike the CSVs it is **already clean and typed** — this section is
about query logic, not string surgery.

| table | grain | columns |
|---|---|---|
| `customers` | customer | `customer_id, signup_date, state, age, segment, annual_income, credit_score_at_signup` |
| `accounts` | account | `account_id, customer_id, product, open_date, credit_limit, apr, status` |
| `transactions` | transaction | `transaction_id, account_id, txn_ts, amount, merchant_category, merchant_id, channel, is_disputed` |
| `statements` | account × month | `statement_id, account_id, statement_month, statement_balance, min_payment_due, payment_made, days_past_due` |

Notes that matter:

- `status` ∈ `{'open', 'closed', 'charged_off'}`.
- `merchant_category` is stored in canonical mixed case here (`'Groceries'`), and can be `NULL`.
- Dates are ISO text: `txn_ts` is `'YYYY-MM-DD HH:MM:SS'`, `statement_month` is the **first of the
  month**, `'YYYY-MM-01'`. `strftime`, `date(x, '+1 month')` and plain string comparison all work.
- `days_past_due` ∈ `{0, 30, 60, 90, 120}`. An account stops producing statements once it hits 120.
- SQLite 3.45 — window functions, CTEs and `FILTER` are all available.

Write each answer as a **single SQL statement** and run it with the `q(...)` helper.

### S1 — Charge-off rate by segment and product  *(~4 min)*

For each `(segment, product)` combination **with at least 50 accounts**, return one row:

`segment, product, n_accounts, n_charged_off, charge_off_rate`

where `charge_off_rate = n_charged_off / n_accounts` **rounded to 4dp**, and `n_charged_off` counts
accounts with `status = 'charged_off'`.

Order by `charge_off_rate` **descending**, then `segment` ascending, then `product` ascending.

In [35]:
# --- S1 ---
s1 = q("""
SELECT *
FROM customers 
""")
check("S1", s1)

--- S1 ----------------------------------------------------------
FAIL S1: missing column(s) ['product', 'n_accounts', 'n_charged_off', 'charge_off_rate']
    S1 score: 0%



### S2 — Everyone's #1 spending category  *(~7 min)*

Using 2025 transactions where `amount > 0` and `merchant_category IS NOT NULL`:

For each customer, find the merchant category they spent the most in (ties broken **alphabetically**
by category name). Then return, **per category**, how many customers have it as their #1:

`merchant_category, n_customers`

Order by `n_customers` descending, then `merchant_category` ascending.

**Use a window function.** A correlated subquery that happens to produce the same numbers is not what
the question is testing — this is exactly the pattern (`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`)
that shows up in nearly every data-scientist SQL screen.

In [37]:
# --- S2 ---
s2 = q("""
SELECT * 
FROM transactions
WHERE amount > 0 AND merchant_category IS NOT NULL
ORDER BY merchant_category
""")
check("S2", s2)

--- S2 ----------------------------------------------------------
FAIL S2: missing column(s) ['n_customers']
    S2 score: 0%



### S3 — Month-over-month portfolio spend  *(~5 min)*

For calendar 2025, total **purchase** spend (`amount > 0`) across the whole portfolio by month:

`month, total_spend, mom_pct_change`

- `month` formatted `'YYYY-MM'`
- `total_spend` rounded to 2dp
- `mom_pct_change` = `100 * (this_month - prev_month) / prev_month`, rounded to 2dp, and
  **NULL for January** (no prior month)

Order by `month` ascending.

In [38]:
# --- S3 ---
s3 = q("""
SELECT month, total_spend, mom_pct_change
FROM transactions
WHERE amount > 0
ORDER BY month
""")
check("S3", s3)

DatabaseError: Execution failed on sql '
SELECT month, total_spend, mom_pct_change
FROM transactions
WHERE amount > 0
ORDER BY month
': no such column: month

### S4 — The 30→60 roll rate  *(~8 min)*

The single most-used credit risk metric: of the accounts that were 30 days past due this month,
what fraction got *worse* next month?

For each `statement_month` from 2025-01 through 2025-11:

- `n_30dpd` — accounts with `days_past_due = 30` in that month
- `n_rolled` — of those, how many had `days_past_due >= 60` in the **immediately following calendar
  month**. An account with no statement in the following month counts as **not** rolled.
- `roll_rate` = `n_rolled / n_30dpd`, rounded to 4dp

Return `month, n_30dpd, n_rolled, roll_rate` with `month` formatted `'YYYY-MM'`, excluding any month
where `n_30dpd = 0`, ordered by `month` ascending.

In [ ]:
# --- S4 ---
s4 = q("""

""")
check("S4", s4)

### S5 — Consecutive delinquency streaks  *(stretch, ~6 min)*

A customer is **delinquent in month M** if **any** of their accounts has `days_past_due > 0` in that
statement month.

For each customer, find the length of their **longest run of consecutive calendar months** in 2025
during which they were delinquent. Return only customers whose longest run is **3 or more**:

`customer_id, longest_streak`

Order by `longest_streak` descending, then `customer_id` ascending.

*This is the "gaps and islands" problem. If you've never seen it: number the delinquent months per
customer with `ROW_NUMBER()`, then note that within a consecutive run, `month - row_number` is
constant. Group on that constant.*

If you are past 30 minutes, skip this and move to Section 3. Come back to it afterwards.

In [ ]:
# --- S5 ---
s5 = q("""

""")
check("S5", s5)

In [39]:
elapsed("end of Section 2 -- target was 50 min cumulative")

[ 33.0 min] end of Section 2 -- target was 50 min cumulative


---
# Section 3 — Applied machine learning  ·  budget 30 minutes

`data/credit_applications.csv` — 20,000 credit card applications from 2023-01 through 2024-12, one row
per application, with the outcome observed 12 months after booking.

**Target:** `default_12m` (1 = went 90+ days past due or charged off within 12 months).

| column | meaning |
|---|---|
| `application_id` | identifier |
| `app_date` | date the application was submitted |
| `requested_product`, `channel` | product applied for; acquisition channel |
| `applicant_age`, `state`, `zip3` | applicant demographics / geography |
| `fico_score` | bureau score at application |
| `annual_income` | stated income (**has missing values**) |
| `dti_ratio` | debt-to-income at application |
| `employment_length_years` | years at current employer (**uses `-1` as an "unknown" sentinel**) |
| `num_inquiries_6m` | credit inquiries in the prior 6 months |
| `revolving_utilization` | revolving utilization (**uses `999` as a bad-data sentinel**) |
| `delinq_2yrs` | delinquencies in the prior 2 years |
| `home_ownership` | RENT / MORTGAGE / OWN / OTHER (**has missing values**) |
| `requested_amount` | credit line requested |
| `assigned_credit_limit` | line actually assigned at underwriting |
| `post_orig_30dpd_count` | count of 30-day delinquencies **after the account was opened** |
| `recovery_amount_usd` | dollars recovered by collections |
| `default_12m` | **target** |

**Read that table carefully before you write any code.** Some of those columns are not legitimate
model inputs, and noticing which ones is the single highest-signal thing you do in this section.

In [41]:
apps = pd.read_csv(f"{DATA}/credit_applications.csv", parse_dates=["app_date"])
print(apps.shape, "| default rate:", round(apps.default_12m.mean(), 4))
apps.head()

(20000, 20) | default rate: 0.086


,application_id,app_date,requested_product,channel,applicant_age,state,zip3,fico_score,annual_income,dti_ratio,employment_length_years,num_inquiries_6m,revolving_utilization,delinq_2yrs,home_ownership,requested_amount,assigned_credit_limit,post_orig_30dpd_count,recovery_amount_usd,default_12m
0,AP908086,2023-01-01,Quicksilver,online,61,OH,676,603.0000,"92,500.0000",0.3242,4.8000,6,0.6848,1,MORTGAGE,"3,800.0000","3,200.0000",0,0.0000,0
1,AP906045,2023-01-01,Venture,online,54,PA,792,700.0000,"45,800.0000",0.2205,2.9000,5,0.7418,0,MORTGAGE,"5,600.0000","2,600.0000",1,0.0000,0
2,AP914612,2023-01-01,Quicksilver,online,40,GA,791,685.0000,"116,500.0000",0.3115,2.3000,2,0.2201,0,MORTGAGE,"2,100.0000","1,300.0000",0,0.0000,0
3,AP900285,2023-01-01,PlatinumSecured,partner,48,TX,226,758.0000,"111,600.0000",0.2738,1.2000,1,0.1225,0,MORTGAGE,"2,700.0000",900.0000,0,0.0000,0
4,AP916690,2023-01-01,Quicksilver,online,44,TX,455,603.0000,"42,700.0000",0.2010,1.8000,6,0.8239,1,RENT,"3,200.0000","1,900.0000",0,0.0000,0


### M1 — A defensible baseline  *(~12 min)*

Build a model you would be willing to defend in a model-risk review.

1. **Drop any feature that would not be knowable at the moment the application is decided.**
2. Handle the coded-missing sentinels (`employment_length_years == -1`, `revolving_utilization == 999`)
   — they are *not* legitimate values.
3. Split **by time**: train on `app_date < '2024-07-01'`, test on `app_date >= '2024-07-01'`.
4. Fit any classifier you like, with preprocessing inside a `Pipeline` (so nothing leaks from test
   into train).
5. Score on the test set with **both** ROC-AUC and PR-AUC (average precision). The positive class is
   ~8% — know why you want both.

**Assign `m1` as a dict:**

| key | type |
|---|---|
| `excluded_features` | list[str] — the columns you dropped as not-available-at-decision |
| `test_roc_auc` | float |
| `test_pr_auc` | float |
| `n_train`, `n_test` | int |

Keep your fitted pipeline and the test-set predicted probabilities — M3 needs them.

In [ ]:
# --- M1 ---


m1 = {
    "excluded_features": ["application_id", "app_date"],
    "test_roc_auc": None,
    "test_pr_auc": None,
    "n_train": None,
    "n_test": None,
}
check("M1", m1)

### M2 — What does a random split cost you?  *(~6 min)*

Refit **the identical pipeline** on a random stratified split with the same test-set size and
`random_state=42`, and compare its ROC-AUC to your honest time-based number.

**Assign `m2`:**

| key | type |
|---|---|
| `random_split_roc_auc` | float |
| `time_split_roc_auc` | float (your M1 number) |
| `optimism` | float — `random - time`, rounded to 4dp |

Then look at the gap and be ready to say, in M4, *why* it exists. It is not an accident of this
dataset — something specific happened in the second half of 2024.

In [ ]:
# --- M2 ---


m2 = {
    "random_split_roc_auc": None,
    "time_split_roc_auc": None,
    "optimism": None,
}
check("M2", m2)

### M3 — Turn the score into a decision  *(~7 min)*

A model with good AUC is worth nothing until someone picks a cutoff. Per approved application, over
12 months:

- an applicant who **does not** default contributes **+\$240**
- an applicant who **does** default costs **−\$1,650**
- a **declined** application contributes **\$0**

Using your **M1 test-set predicted probabilities**, approve when `predicted_probability < t`. Sweep
`t` over a fine grid and find the profit-maximizing cutoff.

**Assign `m3`:**

| key | type | meaning |
|---|---|---|
| `best_threshold` | float | the profit-maximizing `t`, 3dp (**not graded** — it depends on your model's calibration) |
| `expected_profit_per_app` | float | total profit at that cutoff ÷ number of test applications, 2dp |
| `approval_rate_at_best` | float | share of test applications approved at that cutoff, 4dp |
| `profit_at_approve_all` | float | profit per application if you approved everyone, 2dp |

Compare the last two numbers before you move on. That difference is the entire business case for the
model, and it is the number an interviewer will ask you to state out loud.

In [ ]:
# --- M3 ---


m3 = {
    "best_threshold": None,
    "expected_profit_per_app": None,
    "approval_rate_at_best": None,
    "profit_at_approve_all": None,
}
check("M3", m3)

### M4 — The written section  *(~5 min, no code)*

This is not filler. Capital One is a regulated lender, and the open-ended reasoning is weighted at
least as heavily as the code. Write **3–5 sentences each** — specific, not generic. Aim for the level
of detail you would give a skeptical manager, not a textbook.

Fill in the strings below, then grade yourself against the rubric at the end of `SOLUTIONS.md`.

In [42]:
# --- M4 --- (replace each empty string; 3-5 sentences each)

m4 = {

"leakage": """
Which columns did you exclude, and how did you know? Describe the general test you would apply to a
dataset you had never seen before to find this class of problem.
""" "I excluded the applicant ID and application date because those would add noise to the model instead of contributing generalizable patterns to learn from. Application ID itself has no correlation with the target, and neither does the application date. If you wanted to identify more columns like these, we could run SHAP analysis on the model and consult domain experts to get insight into what columns would meaningfully contribute to the model pattern recognition",

"model_choice": """
You get one model in production for a credit decision at a regulated lender. Logistic regression on
binned/WOE features, or gradient boosting? Pick one and defend it. Name the concrete cost of what you
gave up.
""" "Gradient boosting is a type of linear regression, so it is not suited for making binary decisions. However, we would be giving up the enhanced precision offered by gradient boosting",

"imbalance": """
The positive class is about 8%. What did you actually do about it, what metric did you steer on, and
name one commonly recommended technique you deliberately did NOT use and why.
""" "I would have used the .rebalance() method to balance the outcome ratios. ",

"fairness": """
`applicant_age` and `zip3` are both in the file. What is your position on using them in a credit
underwriting model, and what would you do if a permitted variable turned out to be highly correlated
with a protected characteristic?
""" "It is reasonable to use these variables as they can provide high informational value to the model on certain types of applicants. I would ensure proper censoring of personal information linked with the account ID are in place to protect our applicants' privacy",

"monitoring": """
The model ships. What do you monitor, at what cadence, and what specifically triggers a retrain or a
rollback? Note the one thing you cannot measure for 12 months and how you cope in the meantime.
""" "I would monitor the false positive or recall of the model, as that is much costlier than the precision metric. Perhaps once every 10 applicants as that is approximately the break-even point per false positive.",

"next_step": """
You get 30 more minutes. What is the single highest-value thing you add, and why that over the
alternatives?
""",

}

for k, v in m4.items():
    n = len(v.split())
    print(f"{k:14s} {n:4d} words {'  <-- still the prompt text?' if n > 45 else ''}")

leakage         105 words   <-- still the prompt text?
model_choice     66 words   <-- still the prompt text?
imbalance        44 words 
fairness         82 words   <-- still the prompt text?
monitoring       71 words   <-- still the prompt text?
next_step        19 words 


In [43]:
m4

{'leakage': '\nWhich columns did you exclude, and how did you know? Describe the general test you would apply to a\ndataset you had never seen before to find this class of problem.\nI excluded the applicant ID and application date because those would add noise to the model instead of contributing generalizable patterns to learn from. Application ID itself has no correlation with the target, and neither does the application date. If you wanted to identify more columns like these, we could run SHAP analysis on the model and consult domain experts to get insight into what columns would meaningfully contribute to the model pattern recognition',
 'model_choice': '\nYou get one model in production for a credit decision at a regulated lender. Logistic regression on\nbinned/WOE features, or gradient boosting? Pick one and defend it. Name the concrete cost of what you\ngave up.\nGradient boosting is a type of linear regression, so it is not suited for making binary decisions. However, we would 

In [44]:
elapsed("end of Section 3 -- target was 80 min cumulative")
score()

[ 52.0 min] end of Section 3 -- target was 80 min cumulative
task     score   section
----------------------------------------------------
P1         25%   pandas
S1          0%   SQL
S2          0%   SQL
----------------------------------------------------
      pandas:    25%  (1 graded)
         SQL:     0%  (2 graded)
     OVERALL:     8%
M4 is not auto-graded -- score it against the rubric in SOLUTIONS.md.


---
## After the timer stops

1. Note your per-section score from `score()`.
2. Open `SOLUTIONS.md`. Read the worked solution for **everything**, including what you got right —
   the commentary explains *why* the intended approach is the one to reach for under time pressure,
   and flags the trap each task is built around.
3. Grade M4 yourself against the rubric at the end.
4. Re-run any task you failed **from scratch**, not by patching your old cell. The skill being built
   is getting it right the first time.

### Logging this attempt

1. Fill in `NOTES.md` next to this notebook — seed, elapsed time, per-section score, M4 rubric total.
2. Add a row to `PROGRESS.md` at the repo root.
3. Commit: `git add -A && git commit -m "attempt: capital-one-ds #N — 78% (SQL 60%)"`

Save this notebook **with its outputs intact** — the `check()` results are the record of what you
actually did, and they're what you'll diff against next time.

To re-sit later with different numbers: `./bootstrap.sh 12345` in the set folder, then
`./new-attempt.sh capital-one-ds` from the repo root.